# Perfil de um ator

Este notebook mostra, para um único ator, a etapa descrita em `ACTOR_PROFILES.md`: as
falas que entram, o prompt que o modelo recebe, o comando que gera o perfil e a linha que sai
no JSONL. O modelo usado aqui é pequeno o bastante para rodar num notebook comum, e o perfil
que ele escreve serve só para mostrar o formato da saída.

In [1]:
import json
import re
import textwrap
from pathlib import Path

import pandas as pd

## Parâmetros

`MODEL` é o mesmo valor que vai em `--model`: id de um repositório do Hugging Face ou caminho de
uma pasta local. `ACTOR` é o nome exato do campo `actor` no arquivo de entrada.

A saída da demonstração fica em `artifacts/cache/actor_profiles/demo.jsonl`, fora do arquivo da
rodada completa (`artifacts/actor_profiles/actor_profiles.jsonl`). O script pula atores que já
têm linha no arquivo de saída, então um perfil de demonstração gravado lá faria a rodada completa
pular este ator.

In [2]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ACTOR = "Raimundo Santos"
CONFIG_PATH = Path("configs/actor_profiles.toml")
OUTPUT_PATH = Path("artifacts/cache/actor_profiles/demo.jsonl")

## As falas do ator

O arquivo de entrada (`actors_multi_hearing.jsonl`, construído como descrito em
`HEARING_ACTORS.md`) tem uma linha por ator recorrente, com as falas agrupadas por audiência e
cada turno marcado como `speaker` ou `chair` (presidência da sessão). As funções de leitura são
as do próprio gerador, para que o notebook veja exatamente o que o script vê.

In [3]:
from utils.generate_actor_profiles import (
    actor_hearings,
    load_config,
    load_hearing_metadata,
    load_prompts,
)

In [4]:
config = load_config(CONFIG_PATH)
print(config.speeches_path)

artifacts/cache/hearing_actors/actors_multi_hearing.jsonl


In [5]:
with open(config.speeches_path) as f:
    record = next(row for line in f if (row := json.loads(line))["actor"] == ACTOR)

In [6]:
speech_summary = pd.DataFrame(
    [
        {
            "hearing_id": hearing["hearing_id"],
            "turns": len(hearing["turns"]),
            "chair_turns": sum(turn["role"] == "chair" for turn in hearing["turns"]),
            "words": sum(len(turn["text"].split()) for turn in hearing["turns"]),
        }
        for hearing in record["hearings"]
    ]
)
speech_summary

,hearing_id,turns,chair_turns,words
0,166,1,0,395
1,184,1,0,299
2,197,3,2,629


In [7]:
speech_summary[["turns", "chair_turns", "words"]].sum()

turns             5
chair_turns       2
words          1323
dtype: int64

São 5 turnos em 3 audiências, 1.323 palavras no total, e os dois turnos de presidência estão
na audiência 197. O ator foi escolhido por essas duas razões: pouco material deixa a geração
rápida num notebook, e os turnos de presidência mostram a marcação que o template aplica.

## O prompt

O script manda ao modelo duas mensagens: o system prompt fixo
(`prompts/actor_profile/system_profile.md`) e a mensagem do usuário, renderizada pelo template
Jinja2 `user_profile.md.j2` com as falas em ordem cronológica. Cada audiência vem precedida da
data e do assunto, lidos do LDS. As células abaixo refazem essa renderização sem carregar o
modelo. `prompt_version` é o hash dos dois arquivos de prompt e vai gravado em cada perfil.

In [8]:
prompts = load_prompts(config)
print(prompts.version)

8428246d3fae


In [9]:
hearing_metadata = load_hearing_metadata(config)

In [10]:
user_prompt = prompts.user_profile.render(
    actor_label=record["actor"],
    hearings=actor_hearings(record, hearing_metadata),
)
print(user_prompt)

Escreva o perfil de Raimundo Santos a partir das falas abaixo, transcritas de 3 audiência(s) pública(s), em ordem cronológica.

=== Audiência de 09/08/2023 sobre: Revisão do Marco Legal da Inovação para facilitar sua operacionalização ===

[presidência da sessão]
Declaro aberta a presente reunião de audiência pública da Comissão de Ciência, Tecnologia e Inovação, convocada com o objetivo de discutir os impactos e perspectivas de revisão do novo marco legal da inovação.

Esta audiência pública atende o Requerimento nº 20, de 2023, formulado pela ilustre Deputada Luisa Canziani, Presidente desta Comissão, aprovado em reunião deliberativa da Comissão.

Em razão do número de convidados, as apresentações serão divididas em duas Mesas.

Os procedimentos a serem adotados serão os seguintes: cada palestrante terá 10 minutos para fazer a sua exposição, não sendo permitidos apartes durante a exposição.

Encerradas as apresentações das duas Mesas, será concedida a palavra aos Parlamentares inscri

## Como rodar

A geração é o comando abaixo, rodado dentro de `challenge/`. Sem `--actors`, o script processa
todos os atores do arquivo de entrada; sem `--output`, grava em
`artifacts/actor_profiles/actor_profiles.jsonl`; `--model` pode ser omitido se `name` estiver
preenchido na seção `[model]` de `configs/actor_profiles.toml`.

Como o arquivo de saída é aberto em modo append e os atores que já têm linha nele são pulados, a
primeira célula apaga a saída da demonstração, para que cada execução do notebook gere o perfil
de novo. Na segunda, as linhas de log das requisições HTTP ao Hugging Face são filtradas para
deixar a saída legível.

In [11]:
OUTPUT_PATH.unlink(missing_ok=True)

In [12]:
!uv run python -m utils.generate_actor_profiles \
    --model {MODEL} \
    --actors "{ACTOR}" \
    --output {OUTPUT_PATH} \
    2>&1 | grep -v "HTTP Request"

INFO 1 actors selected, prompts 8428246d3fae, loading Qwen/Qwen2.5-1.5B-Instruct


WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 932.37it/s]


INFO [1/1] Raimundo Santos: 3,834 in / 282 out tokens, 20.3s


done: 1 profiles, 0 failures, 0 skipped (resume)
tokens: 3,834 in / 282 out, elapsed 0.3 min


## O resultado

Cada linha do JSONL é um ator. Além do texto em `profile`, a linha registra o modelo, o
`prompt_version`, o volume de material (`n_statements`, `n_hearings`, `hearing_ids`), o tamanho
do prompt e da resposta no tokenizer do modelo e o tempo de geração.

In [13]:
with open(OUTPUT_PATH) as f:
    profile_row = json.loads(f.readline())

In [14]:
pd.Series({key: value for key, value in profile_row.items() if key != "profile"})

actor                          Raimundo Santos
model               Qwen/Qwen2.5-1.5B-Instruct
prompt_version                    8428246d3fae
n_statements                                 5
n_hearings                                   3
hearing_ids                    [166, 184, 197]
input_tokens                              3834
output_tokens                              282
generated_at         2026-09-25T12:05:48+00:00
duration_seconds                          20.3
dtype: object

In [15]:
print(textwrap.fill(profile_row["profile"], width=100))

Raimundo Santos é um participante ativo da Comissão de Ci NodeList de Tecnologia e Inovação. Ele
defende a revisão do Marco Legal da Inovação para facilitar sua operacionalização. Em sua primeira
intervenção, Raimundo Santos destaca a importância da discussão sobre o novo marco legal de
inovação, destacando a necessidade de ajustes para melhorar a aplicação da lei. Ele também aponta a
falta de ciência, tecnologia e inovação como principal problema do país, sugerindo que a aprovação
do requerimento é uma oportunidade para o Brasil avançar neste campo. Em sua segunda intervenção,
Raimundo Santos apoia a criação de um tripé que envolva Governo, empresas e universidades para
promover a inovação no Brasil. Ele enfatiza a importância de investir em inovação e inovar no
desenvolvimento social, garantindo qualidade de vida, alimentação, saúde e paz para aqueles que mais
precisam. Finalmente, em sua terceira intervenção, Raimundo Santos chama a atenção para a
necessidade de proteger a privacida

In [16]:
print(f"palavras no perfil: {len(profile_row['profile'].split())} (limite do prompt: 800)")

palavras no perfil: 173 (limite do prompt: 800)


## Conferências contra as falas

Duas conferências baratas, sem modelo, apontam onde ler o perfil com atenção. Nenhuma das duas
prova que o perfil está correto: distorções escritas com palavras que existem nas falas passam
por ambas.

A primeira trata de nomes e siglas. O system prompt proíbe acrescentar nomes, siglas, órgãos ou
cargos que as falas não trazem, então a célula lista as palavras do perfil com inicial maiúscula
que não aparecem, em nenhuma capitalização, nas falas do ator nem no nome dele. Palavra de início
de frase e paráfrase legítima também podem cair nessa lista, então cada item precisa ser
conferido no texto.

In [17]:
allowed_words = {
    word.lower()
    for hearing in record["hearings"]
    for turn in hearing["turns"]
    for word in re.findall(r"\w+", turn["text"])
} | set(ACTOR.lower().split())

In [18]:
sorted(
    {
        word
        for word in re.findall(r"\w+", profile_row["profile"])
        if word[0].isupper() and word.lower() not in allowed_words
    }
)

['Ci', 'NodeList']

A segunda trata do cabeçalho. O `assunto` de cada audiência vem da matéria jornalística, e o
system prompt manda usá-lo só para situar as falas no tempo e no tema, sem atribuir à pessoa o
que aparece apenas nele. A célula lista os assuntos copiados literalmente no perfil; por ser
cópia exata, ela só pega o vazamento mais grosseiro.

In [19]:
[
    (hearing["date"], hearing["assunto"])
    for hearing in actor_hearings(record, hearing_metadata)
    if hearing["assunto"].lower() in profile_row["profile"].lower()
]

[('09/08/2023',
  'Revisão do Marco Legal da Inovação para facilitar sua operacionalização')]

Na execução salva deste notebook, as duas conferências acusam problemas. `Ci` e `NodeList` são
um trecho sem sentido dentro do nome da comissão ("Comissão de Ci NodeList de Tecnologia e
Inovação"). A segunda frase do perfil ("defende a revisão do Marco Legal da Inovação para
facilitar sua operacionalização") repete o assunto da audiência de 09/08/2023 como posição do
ator. O perfil também abre com "é um participante ativo", a abertura genérica que o system
prompt dá como exemplo do que não fazer. Esses defeitos são do texto que o modelo de
demonstração gerou nesta execução; a qualidade da rodada completa depende do modelo escolhido e
precisa ser medida sobre a saída dele.